# PADA-3DACB — entrenamiento binario, adaptación, baselines y ablaciones

Entrenamiento para **CN vs Impaired** usando la salida esperada de
`pada3dacb_training_artifacts_bidirectional`.

Métodos:
- `source_only`
- `pada3dacb` (prototype + pseudo-labeling)
- `coral`
- `mmd`
- `cdan`
- `aagn`
- `faster_snn`

Ablaciones:
- `no_proto`
- `no_pl`
- `no_cons`
- `no_concept`
- `no_anat`
- `mean_pool`

El mejor checkpoint se selecciona **solo por macro-F1 del source-validation**.
El target de adaptación expone exclusivamente `x`, `subject_id`, `subject_hash`, `cohort`.
El target de evaluación se usa al final para medir transferencia.

In [ ]:
from pathlib import Path
from itertools import cycle
import os, sys, gc, json, time, random, subprocess, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from IPython.display import display
import traceback

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    precision_recall_fscore_support,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    confusion_matrix,
)

warnings.filterwarnings("default")
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Configuración principal

Esta versión incluye un modo `diagnostic` para investigar el colapso binario sin ejecutar los cinco folds:

- folds 0 y 2, seed 42;
- source batches 50/50 CN--Impaired;
- entrada gradual de PADA;
- bloqueo de adaptación si los pseudo-labels aceptados contienen una sola clase;
- durante cada epoch se evalúa únicamente `source_validation`;
- las métricas oficiales siguen usando el criterio canónico de checkpoint;
- se exporta además un análisis con threshold calibrado únicamente en `source_validation`.

Los runs diagnósticos se escriben en un directorio separado y no deben mezclarse con la tabla final de publicación.


In [ ]:
REPOSITORY_URL = "https://github.com/AlejoPatigno/PADA-3DACB.git"
REPOSITORY_REF = "main"
REPO = Path("/kaggle/working/PADA-3DACB")

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

PRECOMPUTED_ROOT = None

# smoke: 1+1 epochs, fold 0.
# diagnostic: folds 0 y 2, seed 42, entrenamiento real.
# full: matriz configurada en FOLDS/SEEDS.
RUN_MODE = "full"
EXECUTE_TRAINING = True

DIRECTIONS = [
    "ADNI_to_OASIS",
    # "OASIS_to_ADNI",
]

FOLDS = [0, 1, 2, 3, 4]
SEEDS = [
    42, 
    # 43,
    # 44,
]

DIAGNOSTIC_FOLDS = [0, 2]
DIAGNOSTIC_SEEDS = [42]

METHODS_TO_RUN = {
    "source_only": True,
    "pada3dacb": True,
    "coral": False,
    "mmd": False,
    "cdan": False,
    "aagn": False,
    "faster_snn": False,
}

ABLATIONS_TO_RUN = {
    "no_proto": False,
    "no_pl": False,
    "no_cons": False,
    "no_concept": False,
    "no_anat": False,
    "mean_pool": False,
}

MODEL = {
    "num_rois": 102,
    "feature_dim": 256,
    "token_dim": 128,
    "base_channels": 32,
    "concept_hidden_dim": 64,
    "token_dropout": 0.20,
    "concept_dropout": 0.20,
}

TRAINING = {
    "warmup_epochs": 10,
    "full_epochs": 50,
    "batch_size": 16,
    "num_workers": 2,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "gradient_clip_norm": 5.0,
    "mixed_precision": True,

    # Cambios diagnósticos contra el colapso.
    "balanced_source_batches": True,
    "pada_adaptation_ramp_epochs": 10,
    "require_pseudo_label_diversity": True,

    # Ahorro de cómputo: target no se evalúa en cada epoch.
    "monitor_target_during_training": False,

    "resume": True,
    "overwrite": False,
    "continue_on_error": True,
}

CORE_LOSS = {
    "classification": 1.0,
    "concept_classification": 1.0,
    "prediction_consistency": 0.1,
    "concept_supervision": 0.5,
    "anatomical_consistency": 0.2,
    "warm_classification": 0.1,
    "warm_concept_classification": 1.0,
    "warm_prediction_consistency": 0.0,
    "warm_concept_supervision": 1.0,
    "warm_anatomical_consistency": 1.0,
    "label_smoothing": 0.1,
}

ADAPTATION = {
    "pada3dacb": {
        "lambda_proto": 1.0,
        "lambda_pl": 0.1,
        # Se conserva 0.90; la diversity gate evita reforzar una sola pseudo-clase.
        "tau_p": 0.90,
        "proto_margin": 1.0,
        "lambda_sep": 0.1,
    },
    "coral": {"weight": 1.0},
    "mmd": {
        "weight": 1.0,
        "bandwidths": [1.0, 2.0, 4.0, 8.0, 16.0],
    },
    "cdan": {
        "weight": 1.0,
        "grl_coefficient": 1.0,
        "hidden_dims": [256, 128],
        "dropout": 0.1,
        "learning_rate": 1e-4,
        "weight_decay": 1e-4,
    },
}

BASELINES = {
    "aagn": {"base_ch": 32, "embed_dim": 128, "dropout": 0.1},
    "faster_snn": {"base_ch": 16, "dropout": 0.1},
    "training": {
        "epochs": 50,
        "learning_rate": 1e-4,
        "weight_decay": 1e-4,
        "gradient_clip_norm": 1.0,
        "mixed_precision": True,
    },
}

# No reutilizar COMPLETED.json de los runs anteriores.
RUNS_ROOT = KAGGLE_WORKING / "pada3dacb_binary_training_runs_diagnostic_v2"

CLASS_ORDER = ["CN", "Impaired"]
CLASS_TO_INDEX = {"CN": 0, "Impaired": 1}
TARGET_SHAPE = (128, 128, 128)

if RUN_MODE not in {"smoke", "diagnostic", "full"}:
    raise ValueError("RUN_MODE debe ser 'smoke', 'diagnostic' o 'full'.")

if RUN_MODE == "smoke":
    EFFECTIVE_FOLDS = [0]
    EFFECTIVE_SEEDS = [42]
    EFFECTIVE_TRAINING = {
        **TRAINING,
        "warmup_epochs": 1,
        "full_epochs": 1,
        "num_workers": 0,
        "pada_adaptation_ramp_epochs": 1,
    }
    EFFECTIVE_BASELINE_TRAINING = {**BASELINES["training"], "epochs": 2}

elif RUN_MODE == "diagnostic":
    EFFECTIVE_FOLDS = list(DIAGNOSTIC_FOLDS)
    EFFECTIVE_SEEDS = list(DIAGNOSTIC_SEEDS)
    EFFECTIVE_TRAINING = dict(TRAINING)
    EFFECTIVE_BASELINE_TRAINING = dict(BASELINES["training"])

else:
    EFFECTIVE_FOLDS = list(FOLDS)
    EFFECTIVE_SEEDS = list(SEEDS)
    EFFECTIVE_TRAINING = dict(TRAINING)
    EFFECTIVE_BASELINE_TRAINING = dict(BASELINES["training"])

print("RUN_MODE:", RUN_MODE)
print("DIRECTIONS:", DIRECTIONS)
print("FOLDS:", EFFECTIVE_FOLDS, "SEEDS:", EFFECTIVE_SEEDS)
print("METHODS:", [k for k,v in METHODS_TO_RUN.items() if v])
print("ABLATIONS:", [k for k,v in ABLATIONS_TO_RUN.items() if v])
print("balanced source:", EFFECTIVE_TRAINING["balanced_source_batches"])
print("PADA ramp epochs:", EFFECTIVE_TRAINING["pada_adaptation_ramp_epochs"])
print("pseudo-label diversity gate:", EFFECTIVE_TRAINING["require_pseudo_label_diversity"])


## 2. Clonar e importar PADA-3DACB

In [ ]:
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPOSITORY_REF, REPOSITORY_URL, str(REPO)],
        check=True,
    )

for name in list(sys.modules):
    if name == "pada3dacb" or name.startswith("pada3dacb."):
        del sys.modules[name]

sys.path.insert(0, str(REPO / "src"))

import pada3dacb
from pada3dacb.models.pada3dacb import build_pada3dacb
from pada3dacb.models.ablations.mean_pooling import build_mean_pool_model
from pada3dacb.models.baselines import build_baseline
from pada3dacb.losses import CorePADA3DACBLoss
from pada3dacb.losses.core_total import CoreLossWeights
from pada3dacb.adaptation import (
    CORALAdaptationMethod, MMDAdaptationMethod, CDANAdaptationMethod,
    DomainDiscriminator, DomainDiscriminatorConfig,
)
from pada3dacb.training.uda_trainer import ProposedPrototypePseudoAdaptationMethod

print("pada3dacb:", pada3dacb.__version__)

## 3. Descubrir output de precomputación y resolver rutas

In [ ]:
def find_precomputed_root():
    if PRECOMPUTED_ROOT is not None:
        p = Path(PRECOMPUTED_ROOT)
        if not p.exists():
            raise FileNotFoundError(p)
        return p.resolve()

    candidates = []
    for p in KAGGLE_INPUT.rglob("artifact_index_binary.csv"):
        root = p.parent.parent
        if (root / "atlas").is_dir() and (root / "splits").is_dir():
            candidates.append(root.resolve())
    candidates = sorted(set(candidates), key=lambda p: (len(p.parts), str(p)))
    if not candidates:
        raise FileNotFoundError(
            "No se encontró artifact_index_binary.csv. Añade el output del precompute como Input."
        )
    return candidates[0]

ARTIFACT_ROOT = find_precomputed_root()
ARTIFACT_INDEX_PATH = ARTIFACT_ROOT / "indices" / "artifact_index_binary.csv"
ROI_MASKS_PATH = ARTIFACT_ROOT / "atlas" / "roi_masks.pt"
ATLAS_METADATA_PATH = ARTIFACT_ROOT / "atlas" / "atlas_metadata.json"
SPLIT_ROOT = ARTIFACT_ROOT / "splits"

artifact_index = pd.read_csv(ARTIFACT_INDEX_PATH)

required = {
    "subject_id","subject_hash","cohort","class_label","label_index",
    "derivative_path","concept_path","jacobian_path"
}
missing = required - set(artifact_index.columns)
if missing:
    raise RuntimeError(f"artifact index incompleto: {sorted(missing)}")

MODEL_READY_ROOTS = sorted(
    set(
        p.resolve()
        for p in KAGGLE_INPUT.rglob("model_ready_complete")
        if p.is_dir() and (p/"ADNI").is_dir() and (p/"OASIS").is_dir()
    )
)

def resolve_artifact_path(v):
    if v is None or pd.isna(v) or not str(v).strip():
        return None
    p = Path(str(v))
    return p if p.is_absolute() else (ARTIFACT_ROOT / p).resolve()

def resolve_mri_path(row):
    original = Path(str(row["derivative_path"]))
    if original.is_file():
        return original.resolve()

    filename = f"{row['subject_id']}_MRI.pt"
    for root in MODEL_READY_ROOTS:
        candidate = root / str(row["cohort"]) / str(row["class_label"]) / filename
        if candidate.is_file():
            return candidate.resolve()

    hits = []
    for root in MODEL_READY_ROOTS:
        hits += list((root / str(row["cohort"])).rglob(filename))
    hits = sorted(set(p.resolve() for p in hits))
    if len(hits) == 1:
        return hits[0]
    if not hits:
        raise FileNotFoundError(f"No MRI para {row['cohort']} {row['subject_id']}")
    raise RuntimeError(f"Múltiples MRI para {row['subject_id']}: {hits}")

resolved = []
for _, row in tqdm(artifact_index.iterrows(), total=len(artifact_index), desc="Resolviendo rutas"):
    d = row.to_dict()
    d["resolved_derivative_path"] = str(resolve_mri_path(row))
    c = resolve_artifact_path(row["concept_path"])
    g = resolve_artifact_path(row["jacobian_path"])
    d["resolved_concept_path"] = str(c) if c is not None else None
    d["resolved_jacobian_path"] = str(g) if g is not None else None
    resolved.append(d)

artifact_index = pd.DataFrame(resolved)

with open(ATLAS_METADATA_PATH, "r", encoding="utf-8") as f:
    atlas_metadata = json.load(f)

ATLAS_HASH = str(atlas_metadata.get("atlas_hash", ""))
ROI_ORDER_HASH = str(atlas_metadata.get("roi_order_hash", ""))

roi_payload = torch.load(ROI_MASKS_PATH, map_location="cpu", weights_only=True)
RAW_ROI_MASKS = roi_payload["roi_masks"].detach().cpu().float().contiguous()
if RAW_ROI_MASKS.ndim != 4 or RAW_ROI_MASKS.shape[0] != MODEL["num_rois"]:
    raise RuntimeError(f"ROI mask contract inválido: {tuple(RAW_ROI_MASKS.shape)}")

print("ARTIFACT_ROOT:", ARTIFACT_ROOT)
display(artifact_index.groupby(["cohort","class_label"]).size().rename("n").reset_index())
print("RAW ROI masks:", tuple(RAW_ROI_MASKS.shape))

## 4. Datasets y splits

In [ ]:
def load_mri(path):
    x = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(x, dict):
        found = [x[k] for k in ("x","image","mri","tensor","volume") if k in x and torch.is_tensor(x[k])]
        if not found:
            raise TypeError(path)
        x = found[0]
    if not torch.is_tensor(x):
        raise TypeError(path)
    x = x.detach().cpu().float()
    if x.ndim == 3:
        x = x.unsqueeze(0)
    if tuple(x.shape) != (1,*TARGET_SHAPE) or not torch.isfinite(x).all():
        raise ValueError(f"MRI inválida {path}: {tuple(x.shape)}")
    return x.contiguous()

def load_vector(path, K):
    v = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(v, dict):
        found = [v[k] for k in ("c_target","concept_target","g_bar","tensor") if k in v and torch.is_tensor(v[k])]
        if not found:
            raise TypeError(path)
        v = found[0]
    v = v.detach().cpu().float().view(-1)
    if tuple(v.shape) != (K,) or not torch.isfinite(v).all():
        raise ValueError(f"Vector inválido {path}: {tuple(v.shape)}")
    return v.contiguous()

class SourceDataset(Dataset):
    def __init__(self, frame, K=102, artifacts=True):
        self.df = frame.reset_index(drop=True)
        self.K = K
        self.artifacts = artifacts
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        item = {
            "x": load_mri(r.resolved_derivative_path),
            "y": torch.tensor(int(r.label_index), dtype=torch.long),
            "subject_id": str(r.subject_id),
            "subject_hash": str(r.subject_hash),
            "cohort": str(r.cohort),
        }
        if self.artifacts:
            cp, gp = r.resolved_concept_path, r.resolved_jacobian_path
            if cp is None or not Path(cp).is_file():
                raise FileNotFoundError(f"c_target faltante: {r.subject_hash}")
            if gp is None or not Path(gp).is_file():
                raise FileNotFoundError(f"g_bar faltante: {r.subject_hash}")
            item["c_target"] = load_vector(cp, self.K)
            item["g_bar"] = load_vector(gp, self.K)
        return item

class TargetAdaptDataset(Dataset):
    def __init__(self, frame): self.df = frame.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return {
            "x": load_mri(r.resolved_derivative_path),
            "subject_id": str(r.subject_id),
            "subject_hash": str(r.subject_hash),
            "cohort": str(r.cohort),
        }

class EvalDataset(SourceDataset):
    def __init__(self, frame): super().__init__(frame, artifacts=False)

ARTIFACT_BY_HASH = {str(r.subject_hash): r for r in artifact_index.itertuples(index=False)}

def frame_for_hashes(values):
    return pd.DataFrame([ARTIFACT_BY_HASH[str(h)]._asdict() for h in values])

def domains(direction):
    src, tgt = direction.split("_to_")
    return src, tgt

def load_fold_frames(direction, fold):
    root = SPLIT_ROOT / direction
    s = pd.read_csv(root / "source_folds.csv")
    t = pd.read_csv(root / "target_split.csv")
    sf = s[s.fold == int(fold)]
    frames = {
        "source_train": frame_for_hashes(sf[sf.partition=="source_train"].subject_hash.astype(str)),
        "source_validation": frame_for_hashes(sf[sf.partition=="source_validation"].subject_hash.astype(str)),
        "target_adaptation": frame_for_hashes(t[t.partition=="target_adaptation"].subject_hash.astype(str)),
        "target_evaluation": frame_for_hashes(t[t.partition=="target_evaluation"].subject_hash.astype(str)),
    }
    if set(frames["source_train"].subject_hash) & set(frames["source_validation"].subject_hash):
        raise RuntimeError("Leakage source train/validation")
    if set(frames["target_adaptation"].subject_hash) & set(frames["target_evaluation"].subject_hash):
        raise RuntimeError("Leakage target adaptation/evaluation")
    return frames

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

class BalancedBinaryBatchSampler:
    """Batch sampler determinista con 50% CN y 50% Impaired por batch."""

    def __init__(self, labels, batch_size, seed):
        labels = np.asarray(labels, dtype=int)
        if batch_size < 2 or batch_size % 2 != 0:
            raise ValueError("balanced_source_batches requiere batch_size par.")
        if not np.isin(labels, [0, 1]).all():
            raise ValueError("El sampler balanceado requiere labels binarios 0/1.")

        self.cn = np.flatnonzero(labels == 0)
        self.impaired = np.flatnonzero(labels == 1)
        if len(self.cn) == 0 or len(self.impaired) == 0:
            raise ValueError("Source train debe contener CN e Impaired.")

        self.batch_size = int(batch_size)
        self.half = self.batch_size // 2
        self.seed = int(seed)
        self.epoch = 0
        self.n_batches = len(labels) // self.batch_size
        if self.n_batches < 1:
            raise ValueError("Source train no alcanza para un batch.")

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return self.n_batches

    def __iter__(self):
        rng = np.random.default_rng(self.seed + 1009 * self.epoch)
        need = self.n_batches * self.half

        cn = rng.choice(
            self.cn,
            size=need,
            replace=need > len(self.cn),
        ).reshape(self.n_batches, self.half)

        impaired = rng.choice(
            self.impaired,
            size=need,
            replace=need > len(self.impaired),
        ).reshape(self.n_batches, self.half)

        for j in range(self.n_batches):
            batch = np.concatenate([cn[j], impaired[j]])
            rng.shuffle(batch)
            yield [int(i) for i in batch]


def make_loader(
    ds, batch_size, shuffle, seed, drop_last, workers, *, batch_sampler=None
):
    common = dict(
        num_workers=workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=workers > 0,
    )

    if batch_sampler is not None:
        return DataLoader(ds, batch_sampler=batch_sampler, **common)

    g = torch.Generator().manual_seed(int(seed))
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        generator=g,
        **common,
    )


def build_loaders(
    frames,
    seed,
    pada=True,
):
    bs = int(
        EFFECTIVE_TRAINING[
            "batch_size"
        ]
    )

    nw = int(
        EFFECTIVE_TRAINING[
            "num_workers"
        ]
    )

    # ============================================================
    # SOURCE TRAIN
    #
    # Este loader SÍ entra al optimizador.
    #
    # PADA necesita:
    # x + y + c_target + g_bar
    # ============================================================

    source_dataset = SourceDataset(
        frames["source_train"],
        artifacts=pada,
    )

    if bool(EFFECTIVE_TRAINING.get("balanced_source_batches", False)):
        source_batch_sampler = BalancedBinaryBatchSampler(
            frames["source_train"]["label_index"].to_numpy(),
            batch_size=bs,
            seed=seed,
        )
        source_train = make_loader(
            source_dataset,
            bs,
            False,
            seed,
            True,
            nw,
            batch_sampler=source_batch_sampler,
        )
    else:
        source_train = make_loader(
            source_dataset,
            bs,
            True,
            seed,
            True,
            nw,
        )

    # ============================================================
    # SOURCE TRAIN EVALUATION
    #
    # Los mismos sujetos, pero:
    # - no shuffle
    # - no drop_last
    # - no necesitamos cargar conceptos/Jacobians
    #
    # Se utiliza exclusivamente para métricas.
    # ============================================================

    source_train_eval = make_loader(
        EvalDataset(
            frames["source_train"]
        ),
        bs,
        False,
        seed,
        False,
        nw,
    )

    # ============================================================
    # SOURCE VALIDATION
    # ============================================================

    source_validation = make_loader(
        EvalDataset(
            frames[
                "source_validation"
            ]
        ),
        bs,
        False,
        seed,
        False,
        nw,
    )

    # ============================================================
    # TARGET ADAPTATION
    #
    # ESTE es el loader que entra al entrenamiento UDA.
    #
    # Importante:
    # NO contiene y.
    # NO contiene c_target.
    # NO contiene g_bar.
    # ============================================================

    target_adaptation = make_loader(
        TargetAdaptDataset(
            frames[
                "target_adaptation"
            ]
        ),
        bs,
        True,
        seed + 1,
        True,
        nw,
    )

    # ============================================================
    # TARGET TRAIN DIAGNOSTIC
    #
    # Exactamente los mismos sujetos de target_adaptation,
    # pero con labels para CALCULAR MÉTRICAS SOLAMENTE.
    #
    # Nunca se pasa al optimizador.
    # ============================================================

    target_train_diagnostic = make_loader(
        EvalDataset(
            frames[
                "target_adaptation"
            ]
        ),
        bs,
        False,
        seed,
        False,
        nw,
    )

    # ============================================================
    # TARGET VALIDATION
    #
    # Es el actual target_evaluation.
    # ============================================================

    target_validation = make_loader(
        EvalDataset(
            frames[
                "target_evaluation"
            ]
        ),
        bs,
        False,
        seed,
        False,
        nw,
    )

    return {

        "source_train":
            source_train,

        "source_train_eval":
            source_train_eval,

        "source_validation":
            source_validation,

        "target_adaptation":
            target_adaptation,

        "target_train_diagnostic":
            target_train_diagnostic,

        "target_validation":
            target_validation,

        # Alias por compatibilidad
        "target_evaluation":
            target_validation,
    }

## 5. Modelos, máscaras ROI, pérdida y adaptación

In [ ]:
def pada_config():
    return {
        "task_id":"cn_vs_impaired",
        "task_type":"binary_classification",
        "class_order":CLASS_ORDER,
        "class_ids":CLASS_TO_INDEX,
        "model":{
            "name":"PADA-3DACB","contextual_encoder":False,"input_channels":1,
            "num_classes":2,"num_rois":MODEL["num_rois"],
            "encoder":{"base_channels":MODEL["base_channels"],"output_channels":MODEL["feature_dim"]},
            "tokenizer":{"feature_dim":MODEL["feature_dim"],"token_dim":MODEL["token_dim"]},
            "token_processing":{"dropout":MODEL["token_dropout"]},
            "concept_bottleneck":{"hidden_dim":MODEL["concept_hidden_dim"],"dropout":MODEL["concept_dropout"]},
        }
    }

def build_pada(mean_pool=False):
    if mean_pool:
        return build_mean_pool_model(
            num_rois=MODEL["num_rois"], feature_dim=MODEL["feature_dim"],
            token_dim=MODEL["token_dim"], num_classes=2,
            base_channels=MODEL["base_channels"],
            concept_hidden_dim=MODEL["concept_hidden_dim"],
            token_dropout=MODEL["token_dropout"], concept_dropout=MODEL["concept_dropout"],
        )
    return build_pada3dacb(pada_config())

probe = build_pada()
FEATURE_SHAPE = tuple(int(x) for x in probe.encoder.infer_output_shape((1,1,*TARGET_SHAPE))[-3:])
del probe

def prepare_masks(masks, shape):
    masks = masks.detach().cpu().float()
    if tuple(masks.shape[-3:]) != tuple(shape):
        out=[]
        for k in range(masks.shape[0]):
            out.append(F.adaptive_avg_pool3d(masks[k:k+1].unsqueeze(1), shape)[0,0])
        masks = torch.stack(out)
    flat=masks.flatten(1)
    sums=flat.sum(1,keepdim=True)
    bad=(sums<=0).nonzero(as_tuple=False).flatten().tolist()
    if bad: raise RuntimeError(f"ROI vacías: {bad}")
    return (flat/sums.clamp_min(1e-8)).view_as(masks).contiguous()

FEATURE_ROI_MASKS = prepare_masks(RAW_ROI_MASKS, FEATURE_SHAPE)

def build_core(ablation=None):
    w = dict(CORE_LOSS)
    label_smoothing = w.pop("label_smoothing")
    if ablation == "no_cons":
        w["prediction_consistency"] = 0.0
    elif ablation == "no_concept":
        w["concept_classification"] = 0.0
        w["concept_supervision"] = 0.0
    elif ablation == "no_anat":
        w["anatomical_consistency"] = 0.0
    return CorePADA3DACBLoss(
        num_rois=MODEL["num_rois"],
        weights=CoreLossWeights(**w),
        label_smoothing=label_smoothing,
    )

def build_adaptation(method, ablation=None):
    if method=="source_only": return None,0.0,None
    if method=="pada3dacb":
        cfg=dict(ADAPTATION["pada3dacb"])
        if ablation=="no_proto": cfg["lambda_proto"]=0.0
        if ablation=="no_pl": cfg["lambda_pl"]=0.0
        return ProposedPrototypePseudoAdaptationMethod(**cfg, num_classes=2),1.0,None
    if method=="coral":
        return CORALAdaptationMethod(),float(ADAPTATION["coral"]["weight"]),None
    if method=="mmd":
        return MMDAdaptationMethod(ADAPTATION["mmd"]["bandwidths"]),float(ADAPTATION["mmd"]["weight"]),None
    if method=="cdan":
        cfg=ADAPTATION["cdan"]
        disc=DomainDiscriminator(DomainDiscriminatorConfig(
            input_dim=MODEL["token_dim"]*2,
            hidden_dims=cfg["hidden_dims"], activation="relu", dropout=cfg["dropout"], output_dim=1,
        ))
        return CDANAdaptationMethod(disc,cfg["grl_coefficient"]),float(cfg["weight"]),disc
    raise ValueError(method)

def build_binary_baseline(name):
    cfg={"task_id":"cn_vs_impaired","task_type":"binary_classification","class_order":CLASS_ORDER,"class_ids":CLASS_TO_INDEX}
    cfg.update(BASELINES[name])
    if name=="aagn": cfg["roi_masks"]=FEATURE_ROI_MASKS
    return build_baseline(name,cfg)

print("feature shape:", FEATURE_SHAPE)
print("feature ROI masks:", tuple(FEATURE_ROI_MASKS.shape))

## 6. Métricas, evaluación y checkpoints

In [ ]:
def metrics_binary(
    y,
    probabilities,
    *,
    decision_threshold=0.5,
):
    y = np.asarray(
        y,
        dtype=int,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    if probabilities.ndim != 2:
        raise ValueError(
            "probabilities debe ser rank-2."
        )

    if probabilities.shape[1] != 2:
        raise ValueError(
            "La tarea binaria requiere probabilities (N,2)."
        )

    if len(y) != len(probabilities):
        raise ValueError(
            "y y probabilities tienen diferente longitud."
        )

    if len(y) == 0:
        raise ValueError(
            "No se pueden calcular métricas sobre un split vacío."
        )

    if not np.isfinite(
        probabilities
    ).all():
        raise ValueError(
            "Las probabilidades contienen NaN/Inf."
        )

    # ============================================================
    # NORMALIZACIÓN DEFENSIVA
    # ============================================================

    row_sums = probabilities.sum(
        axis=1,
        keepdims=True,
    )

    if np.any(row_sums <= 0):
        raise ValueError(
            "Hay filas de probabilidades con suma <= 0."
        )

    probabilities = (
        probabilities
        /
        row_sums
    )

    decision_threshold = float(decision_threshold)
    if not 0.0 <= decision_threshold <= 1.0:
        raise ValueError("decision_threshold debe estar en [0,1].")

    pred = (
        probabilities[:, 1] >= decision_threshold
    ).astype(int)

    # ============================================================
    # CONFUSION MATRIX
    #
    #              predicted
    #              CN   Impaired
    #
    # true CN      TN      FP
    # true Imp     FN      TP
    # ============================================================

    cm = confusion_matrix(
        y,
        pred,
        labels=[0, 1],
    )

    tn, fp, fn, tp = (
        cm.ravel()
    )

    # ============================================================
    # MÉTRICAS POR CLASE
    # ============================================================

    precision_per_class, recall_per_class, f1_per_class, support = (
        precision_recall_fscore_support(
            y,
            pred,
            labels=[0, 1],
            zero_division=0,
        )
    )

    # ============================================================
    # ERRORES
    # ============================================================

    n = len(y)

    n_errors = int(
        (pred != y).sum()
    )

    error_rate = (
        n_errors / n
    )

    false_positive_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else np.nan
    )

    false_negative_rate = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else np.nan
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    ppv = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else np.nan
    )

    npv = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )

    # ============================================================
    # MÉTRICAS PRINCIPALES
    # ============================================================

    out = {

        "decision_threshold": float(decision_threshold),

        # --------------------------------------------------------
        # Tamaño
        # --------------------------------------------------------

        "n":
            int(n),

        "n_cn":
            int((y == 0).sum()),

        "n_impaired":
            int((y == 1).sum()),


        # --------------------------------------------------------
        # Accuracy
        # --------------------------------------------------------

        "accuracy":
            float(
                accuracy_score(
                    y,
                    pred,
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    y,
                    pred,
                )
            ),


        # --------------------------------------------------------
        # Precision
        # --------------------------------------------------------

        "macro_precision":
            float(
                precision_score(
                    y,
                    pred,
                    average="macro",
                    zero_division=0,
                )
            ),

        "weighted_precision":
            float(
                precision_score(
                    y,
                    pred,
                    average="weighted",
                    zero_division=0,
                )
            ),


        # --------------------------------------------------------
        # Recall
        # --------------------------------------------------------

        "macro_recall":
            float(
                recall_score(
                    y,
                    pred,
                    average="macro",
                    zero_division=0,
                )
            ),

        "weighted_recall":
            float(
                recall_score(
                    y,
                    pred,
                    average="weighted",
                    zero_division=0,
                )
            ),


        # --------------------------------------------------------
        # F1
        # --------------------------------------------------------

        "macro_f1":
            float(
                f1_score(
                    y,
                    pred,
                    average="macro",
                    zero_division=0,
                )
            ),

        "weighted_f1":
            float(
                f1_score(
                    y,
                    pred,
                    average="weighted",
                    zero_division=0,
                )
            ),


        # --------------------------------------------------------
        # CN
        # --------------------------------------------------------

        "precision_cn":
            float(
                precision_per_class[0]
            ),

        "recall_cn":
            float(
                recall_per_class[0]
            ),

        "f1_cn":
            float(
                f1_per_class[0]
            ),

        "support_cn":
            int(
                support[0]
            ),


        # --------------------------------------------------------
        # Impaired
        # --------------------------------------------------------

        "precision_impaired":
            float(
                precision_per_class[1]
            ),

        "recall_impaired":
            float(
                recall_per_class[1]
            ),

        "f1_impaired":
            float(
                f1_per_class[1]
            ),

        "support_impaired":
            int(
                support[1]
            ),


        # --------------------------------------------------------
        # Métricas clínicas
        # --------------------------------------------------------

        "sensitivity":
            (
                None
                if np.isnan(sensitivity)
                else float(sensitivity)
            ),

        "specificity":
            (
                None
                if np.isnan(specificity)
                else float(specificity)
            ),

        "ppv":
            (
                None
                if np.isnan(ppv)
                else float(ppv)
            ),

        "npv":
            (
                None
                if np.isnan(npv)
                else float(npv)
            ),


        # --------------------------------------------------------
        # Agreement
        # --------------------------------------------------------

        "mcc":
            float(
                matthews_corrcoef(
                    y,
                    pred,
                )
            ),

        "cohen_kappa":
            float(
                cohen_kappa_score(
                    y,
                    pred,
                )
            ),


        # --------------------------------------------------------
        # Probabilistic errors
        # --------------------------------------------------------

        "log_loss":
            float(
                log_loss(
                    y,
                    np.clip(
                        probabilities,
                        1e-7,
                        1 - 1e-7,
                    ),
                    labels=[0, 1],
                )
            ),

        "brier_score":
            float(
                brier_score_loss(
                    y,
                    probabilities[:, 1],
                )
            ),


        # --------------------------------------------------------
        # Error counts
        # --------------------------------------------------------

        "n_errors":
            int(n_errors),

        "error_rate":
            float(error_rate),

        "false_positives":
            int(fp),

        "false_negatives":
            int(fn),

        "true_positives":
            int(tp),

        "true_negatives":
            int(tn),

        "false_positive_rate":
            (
                None
                if np.isnan(
                    false_positive_rate
                )
                else float(
                    false_positive_rate
                )
            ),

        "false_negative_rate":
            (
                None
                if np.isnan(
                    false_negative_rate
                )
                else float(
                    false_negative_rate
                )
            ),

        "confusion_matrix":
            cm.astype(int).tolist(),
    }

    # ============================================================
    # ROC / PR
    # ============================================================

    if len(
        np.unique(y)
    ) == 2:

        out["roc_auc"] = float(
            roc_auc_score(
                y,
                probabilities[:, 1],
            )
        )

        out["pr_auc"] = float(
            average_precision_score(
                y,
                probabilities[:, 1],
            )
        )

    else:

        out["roc_auc"] = None
        out["pr_auc"] = None

    return out

@torch.no_grad()
def eval_model(
    model,
    loader,
    device,
    roi_masks=None,
    baseline=False,
    *,
    decision_threshold=0.5,
):
    model.eval()

    ys = []
    probabilities = []
    rows = []

    for batch in loader:

        x = batch["x"].to(
            device,
            non_blocking=True,
        )

        y = (
            batch["y"]
            .cpu()
            .numpy()
        )

        # ========================================================
        # FORWARD
        # ========================================================

        if baseline:

            output = model(x)

            logits = (
                output["logits"]
                if isinstance(
                    output,
                    dict,
                )
                else output
            )

        else:

            output = model(
                x,
                roi_masks,
            )

            # Seguimos usando el head conceptual
            # como output predictivo principal.
            logits = (
                output.concept_logits
            )

        p = (
            torch.softmax(
                logits,
                dim=-1,
            )
            .detach()
            .cpu()
            .numpy()
        )

        pred = (
            p[:, 1] >= float(decision_threshold)
        ).astype(int)

        ys.extend(
            y.tolist()
        )

        probabilities.extend(
            p.tolist()
        )

        # ========================================================
        # REGISTRO SUJETO A SUJETO
        # ========================================================

        for i in range(
            len(y)
        ):

            y_true = int(
                y[i]
            )

            y_pred = int(
                pred[i]
            )

            is_error = (
                y_true != y_pred
            )

            if (
                y_true == 0
                and
                y_pred == 1
            ):

                error_type = (
                    "FALSE_POSITIVE"
                )

            elif (
                y_true == 1
                and
                y_pred == 0
            ):

                error_type = (
                    "FALSE_NEGATIVE"
                )

            else:

                error_type = (
                    "CORRECT"
                )

            true_probability = float(
                p[
                    i,
                    y_true,
                ]
            )

            predicted_probability = float(
                p[
                    i,
                    y_pred,
                ]
            )

            rows.append({

                "subject_id":
                    batch[
                        "subject_id"
                    ][i],

                "subject_hash":
                    batch[
                        "subject_hash"
                    ][i],

                "cohort":
                    batch[
                        "cohort"
                    ][i],

                "y_true":
                    y_true,

                "true_label":
                    CLASS_ORDER[
                        y_true
                    ],

                "y_pred":
                    y_pred,

                "predicted_label":
                    CLASS_ORDER[
                        y_pred
                    ],

                "prob_cn":
                    float(
                        p[i, 0]
                    ),

                "prob_impaired":
                    float(
                        p[i, 1]
                    ),

                "true_class_probability":
                    true_probability,

                "predicted_class_probability":
                    predicted_probability,

                "confidence":
                    float(
                        p[i].max()
                    ),

                "probability_margin":
                    float(
                        abs(
                            p[i, 1]
                            -
                            p[i, 0]
                        )
                    ),

                "is_error":
                    bool(
                        is_error
                    ),

                "error_type":
                    error_type,
            })

    metrics = metrics_binary(
        ys,
        probabilities,
        decision_threshold=decision_threshold,
    )

    predictions = pd.DataFrame(
        rows
    )

    return (
        metrics,
        predictions,
    )

def find_source_validation_threshold(predictions):
    """Selecciona threshold usando solo source_validation."""
    required = {"y_true", "prob_cn", "prob_impaired"}
    missing = required - set(predictions.columns)
    if missing:
        raise ValueError(f"Faltan columnas para calibración: {sorted(missing)}")

    y = predictions["y_true"].to_numpy(dtype=int)
    p = predictions["prob_impaired"].to_numpy(dtype=float)
    unique = np.unique(np.clip(p, 0.0, 1.0))

    candidates = [0.0, 0.5, 1.0]
    candidates += unique.tolist()
    if len(unique) > 1:
        candidates += (0.5 * (unique[:-1] + unique[1:])).tolist()

    best = None
    for threshold in sorted(set(float(t) for t in candidates)):
        pred = (p >= threshold).astype(int)
        ba = float(balanced_accuracy_score(y, pred))
        mf1 = float(f1_score(y, pred, average="macro", zero_division=0))
        key = (ba, mf1, -abs(threshold - 0.5))
        if best is None or key > best["key"]:
            best = {
                "key": key,
                "threshold": float(threshold),
                "source_validation_balanced_accuracy": ba,
                "source_validation_macro_f1": mf1,
            }

    best.pop("key")
    return best


def metrics_from_prediction_table(predictions, threshold):
    return metrics_binary(
        predictions["y_true"].to_numpy(dtype=int),
        predictions[["prob_cn", "prob_impaired"]].to_numpy(dtype=float),
        decision_threshold=float(threshold),
    )


def atomic_save(payload,path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+".tmp"); torch.save(payload,tmp); os.replace(tmp,path)

def save_ckpt(path, model, optimizer, epoch, best_f1, history, config, discriminator=None):
    payload={
        "model_state_dict":model.state_dict(),"optimizer_state_dict":optimizer.state_dict(),
        "epoch":int(epoch),"best_source_macro_f1":float(best_f1),"history":history,
        "config":config,"task_id":"cn_vs_impaired","class_order":CLASS_ORDER,
        "atlas_hash":ATLAS_HASH,"roi_order_hash":ROI_ORDER_HASH,
    }
    if discriminator is not None: payload["discriminator_state_dict"]=discriminator.state_dict()
    atomic_save(payload,path)

def load_ckpt(path,model,device,optimizer=None,discriminator=None):
    p=torch.load(path,map_location=device,weights_only=False)
    model.load_state_dict(p["model_state_dict"],strict=True)
    if optimizer is not None: optimizer.load_state_dict(p["optimizer_state_dict"])
    if discriminator is not None and "discriminator_state_dict" in p:
        discriminator.load_state_dict(p["discriminator_state_dict"],strict=True)
    return p

In [ ]:
def evaluate_all_splits(
    model,
    loaders,
    device,
    *,
    roi_masks=None,
    baseline=False,
):
    results = {}
    predictions = {}

    split_mapping = {

        "source_train":
            "source_train_eval",

        "source_validation":
            "source_validation",

        "target_train_diagnostic":
            "target_train_diagnostic",

        "target_validation":
            "target_validation",
    }

    for output_name, loader_name in (
        split_mapping.items()
    ):

        metrics, pred = eval_model(
            model,
            loaders[
                loader_name
            ],
            device,
            roi_masks=roi_masks,
            baseline=baseline,
        )

        results[
            output_name
        ] = metrics

        predictions[
            output_name
        ] = pred

    return (
        results,
        predictions,
    )

def flatten_split_metrics(
    split_metrics,
):
    flat = {}

    for split_name, metrics in (
        split_metrics.items()
    ):

        for metric_name, value in (
            metrics.items()
        ):

            # Confusion matrix se conserva
            # en JSON, no en CSV.
            if metric_name == (
                "confusion_matrix"
            ):
                continue

            flat[
                f"{split_name}/{metric_name}"
            ] = value

    return flat


def append_jsonl(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        path,
        "a",
        encoding="utf-8",
    ) as f:

        f.write(
            json.dumps(
                payload,
                ensure_ascii=False,
            )
            + "\n"
        )

## 7. Readiness por dirección

In [ ]:
def exists(v): return v is not None and not pd.isna(v) and Path(str(v)).is_file()

ready=[]
for direction in DIRECTIONS:
    src,tgt=domains(direction)
    s=artifact_index[artifact_index.cohort==src]
    t=artifact_index[artifact_index.cohort==tgt]
    smri=s.resolved_derivative_path.map(lambda x:Path(x).is_file()).sum()
    scon=s.resolved_concept_path.map(exists).sum()
    sjac=s.resolved_jacobian_path.map(exists).sum()
    tmri=t.resolved_derivative_path.map(lambda x:Path(x).is_file()).sum()
    ready.append({
        "direction":direction,"source_n":len(s),"source_mri":int(smri),
        "source_concepts":int(scon),"source_jacobians":int(sjac),
        "target_n":len(t),"target_mri":int(tmri),
        "pada_ready":smri==len(s) and scon==len(s) and sjac==len(s) and tmri==len(t),
        "baseline_ready":smri==len(s) and tmri==len(t),
    })
readiness_df=pd.DataFrame(ready)
display(readiness_df)

## 8. Entrenador PADA / Source-only / CORAL / MMD / CDAN / ablaciones

In [ ]:
def run_pada(direction,fold,seed,method,ablation=None):
    seed_all(seed)
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    run_name=method if ablation is None else f"ablation_{ablation}"
    run_dir=RUNS_ROOT/direction/run_name/f"seed_{seed}"/f"fold_{fold}"
    run_dir.mkdir(parents=True,exist_ok=True)
    marker=run_dir/"COMPLETED.json"
    if marker.is_file() and not EFFECTIVE_TRAINING["overwrite"]:
        x=json.loads(marker.read_text()); x["reused"]=True; return x

    frames=load_fold_frames(direction,fold)
    loaders=build_loaders(frames,seed,pada=True)
    model=build_pada(mean_pool=(ablation=="mean_pool")).to(device)
    masks=FEATURE_ROI_MASKS.to(device)
    core=build_core(ablation).to(device)
    adapt,adapt_weight,disc=build_adaptation(method,ablation)
    if disc is not None: disc.to(device)

    if method=="cdan":
        c=ADAPTATION["cdan"]
        optimizer=torch.optim.AdamW([
            {"params":model.parameters(),"lr":EFFECTIVE_TRAINING["learning_rate"],"weight_decay":EFFECTIVE_TRAINING["weight_decay"]},
            {"params":disc.parameters(),"lr":c["learning_rate"],"weight_decay":c["weight_decay"]},
        ])
    else:
        optimizer=torch.optim.AdamW(model.parameters(),lr=EFFECTIVE_TRAINING["learning_rate"],weight_decay=EFFECTIVE_TRAINING["weight_decay"])

    amp=bool(EFFECTIVE_TRAINING["mixed_precision"] and device.type=="cuda")
    scaler=torch.amp.GradScaler("cuda",enabled=amp)
    warm=int(EFFECTIVE_TRAINING["warmup_epochs"]); full=int(EFFECTIVE_TRAINING["full_epochs"]); total=warm+full
    # ================================================================
    # CHECKPOINTS INDEPENDIENTES PARA WARM Y FULL
    # ================================================================
    
    history = []
    
    best_warm = -np.inf
    best_full = -np.inf
    
    best_warm_epoch = None
    best_full_epoch = None
    
    start = 1
    cfg={"direction":direction,"fold":fold,"seed":seed,"method":method,"ablation":ablation,
         "model":MODEL,"training":EFFECTIVE_TRAINING,"core_loss":CORE_LOSS,"adaptation":ADAPTATION,
         "prediction_head":"concept","class_order":CLASS_ORDER}
    (run_dir/"resolved_config.json").write_text(json.dumps(cfg,indent=2),encoding="utf-8")
    last = run_dir / "checkpoint_last.pt"
    
    if (
        EFFECTIVE_TRAINING["resume"]
        and last.is_file()
        and not EFFECTIVE_TRAINING["overwrite"]
    ):
    
        p = load_ckpt(
            last,
            model,
            device,
            optimizer,
            disc,
        )
    
        start = int(
            p["epoch"]
        ) + 1
    
        history = list(
            p.get(
                "history",
                [],
            )
        )
    
        # ============================================================
        # RECONSTRUIR LOS DOS BEST DESDE EL HISTORY
        #
        # Esto evita depender del antiguo best_source_macro_f1,
        # que mezclaba warm y full.
        # ============================================================
    
        warm_candidates = []
        full_candidates = []
    
        for row in history:
    
            stage_row = row.get(
                "stage"
            )
    
            f1 = row.get(
                "source_validation/macro_f1"
            )
    
            epoch_row = row.get(
                "epoch"
            )
    
            if (
                f1 is None
                or epoch_row is None
            ):
                continue
    
            try:
                f1 = float(f1)
                epoch_row = int(epoch_row)
            except Exception:
                continue
    
            if not np.isfinite(f1):
                continue
    
            if stage_row == "warm":
    
                warm_candidates.append(
                    (
                        f1,
                        epoch_row,
                    )
                )
    
            elif stage_row == "full":
    
                full_candidates.append(
                    (
                        f1,
                        epoch_row,
                    )
                )
    

        if warm_candidates:
    
            (
                best_warm,
                best_warm_epoch,
            ) = max(
                warm_candidates,
                key=lambda x: x[0],
            )
    
    
        if full_candidates:
    
            (
                best_full,
                best_full_epoch,
            ) = max(
                full_candidates,
                key=lambda x: x[0],
            )
    
    
        print(
            f"RESUME epoch={start} | "
            f"best_warm={best_warm:.4f} "
            f"(epoch={best_warm_epoch}) | "
            f"best_full={best_full:.4f} "
            f"(epoch={best_full_epoch})"
        )
    for epoch in range(start,total+1):
        stage="warm" if epoch<=warm else "full"
        model.train()
        if disc is not None:
            disc.train()

        source_batch_sampler = getattr(
            loaders["source_train"], "batch_sampler", None
        )
        if hasattr(source_batch_sampler, "set_epoch"):
            source_batch_sampler.set_epoch(epoch)

        full_epoch_index = max(0, epoch - warm)
        ramp_epochs = max(
            1,
            int(EFFECTIVE_TRAINING.get("pada_adaptation_ramp_epochs", 1)),
        )
        pada_ramp = (
            min(1.0, full_epoch_index / ramp_epochs)
            if stage == "full"
            else 0.0
        )

        sums={
            "total":0.0,
            "core":0.0,
            "adapt":0.0,
            "adapt_raw":0.0,
            "accepted_cn":0,
            "accepted_impaired":0,
            "rejected_target":0,
            "adapt_active_batches":0,
            "adapt_blocked_single_class_batches":0,
        }
        n=0
        tgt_iter=iter(loaders["target_adaptation"]) if method!="source_only" and stage=="full" else None

        for sb in tqdm(loaders["source_train"],desc=f"{run_name} {direction} f{fold} e{epoch}/{total}",leave=False):
            optimizer.zero_grad(set_to_none=True)
            x=sb["x"].to(device,non_blocking=True); y=sb["y"].to(device,non_blocking=True)
            ctar=sb["c_target"].to(device,non_blocking=True); gbar=sb["g_bar"].to(device,non_blocking=True)
            with torch.autocast(device_type=device.type,enabled=amp):
                sout=model(x,masks); lo=core(sout,y,ctar,gbar,stage=stage); loss=lo.total
                weighted=loss.detach()*0
                if tgt_iter is not None:
                    try:
                        tb=next(tgt_iter)
                    except StopIteration:
                        tgt_iter=iter(loaders["target_adaptation"])
                        tb=next(tgt_iter)

                    if set(tb)!={"x","subject_id","subject_hash","cohort"}:
                        raise RuntimeError(f"Target adaptation firewall: {sorted(tb)}")
                    tout=model(tb["x"].to(device,non_blocking=True),masks)
                    if method=="pada3dacb":
                        ao=adapt.compute(sout,tout,"full",labels_src=y)
                        raw_weighted=ao.total

                        with torch.no_grad():
                            target_prob=torch.softmax(
                                tout.concept_logits.detach(),
                                dim=-1,
                            )
                            target_conf,target_pseudo=target_prob.max(dim=-1)
                            accepted=target_conf >= float(
                                ADAPTATION["pada3dacb"]["tau_p"]
                            )
                            accepted_cn=int(
                                (accepted & (target_pseudo==0)).sum().item()
                            )
                            accepted_impaired=int(
                                (accepted & (target_pseudo==1)).sum().item()
                            )
                            rejected_target=int((~accepted).sum().item())

                        pseudo_diverse=(
                            accepted_cn>0 and accepted_impaired>0
                        )
                        require_diversity=bool(
                            EFFECTIVE_TRAINING.get(
                                "require_pseudo_label_diversity",
                                False,
                            )
                        )
                        gate_open=pseudo_diverse or not require_diversity

                        if gate_open:
                            weighted=raw_weighted*float(pada_ramp)
                            sums["adapt_active_batches"]+=1
                        else:
                            weighted=raw_weighted*0.0
                            sums["adapt_blocked_single_class_batches"]+=1

                        sums["adapt_raw"]+=float(raw_weighted.detach().cpu())
                        sums["accepted_cn"]+=accepted_cn
                        sums["accepted_impaired"]+=accepted_impaired
                        sums["rejected_target"]+=rejected_target

                    else:
                        ao=adapt.compute(sout,tout,"full")
                        weighted=float(adapt_weight)*ao.total
                        sums["adapt_raw"]+=float(weighted.detach().cpu())

                    loss=loss+weighted
            if not torch.isfinite(loss): raise RuntimeError("Loss no finita")
            params=list(model.parameters())+(list(disc.parameters()) if disc is not None else [])
            if scaler.is_enabled():
                scaler.scale(loss).backward(); scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params,EFFECTIVE_TRAINING["gradient_clip_norm"])
                scaler.step(optimizer); scaler.update()
            else:
                loss.backward(); torch.nn.utils.clip_grad_norm_(params,EFFECTIVE_TRAINING["gradient_clip_norm"]); optimizer.step()
            sums["total"]+=float(loss.detach().cpu()); sums["core"]+=float(lo.total.detach().cpu()); sums["adapt"]+=float(weighted.detach().cpu()); n+=1

        # ================================================================
        # EVALUACIÓN POR EPOCH: SOLO SOURCE VALIDATION
        # ================================================================

        source_val_metrics, _ = eval_model(
            model,
            loaders["source_validation"],
            device,
            roi_masks=masks,
            baseline=False,
        )
        epoch_metrics={"source_validation":source_val_metrics}

        total_accepted=sums["accepted_cn"]+sums["accepted_impaired"]
        total_target_seen=total_accepted+sums["rejected_target"]

        row={
            "epoch":int(epoch),
            "stage":stage,
            "train_total_loss":sums["total"]/max(n,1),
            "train_core_loss":sums["core"]/max(n,1),
            "train_adaptation_loss":sums["adapt"]/max(n,1),
            "train_adaptation_raw_loss":sums["adapt_raw"]/max(n,1),
            "pada_adaptation_ramp":float(pada_ramp),
            "target_pseudo_accepted_cn":int(sums["accepted_cn"]),
            "target_pseudo_accepted_impaired":int(sums["accepted_impaired"]),
            "target_pseudo_rejected":int(sums["rejected_target"]),
            "target_pseudo_acceptance_rate":(
                float(total_accepted/total_target_seen)
                if total_target_seen>0 else 0.0
            ),
            "adapt_active_batches":int(sums["adapt_active_batches"]),
            "adapt_blocked_single_class_batches":int(
                sums["adapt_blocked_single_class_batches"]
            ),
            **flatten_split_metrics(epoch_metrics),
        }
        history.append(row)

        append_jsonl(
            run_dir/"epoch_metrics.jsonl",
            {
                "epoch":int(epoch),
                "stage":stage,
                "metrics":epoch_metrics,
                "adaptation_diagnostics":{
                    "pada_adaptation_ramp":float(pada_ramp),
                    "accepted_cn":int(sums["accepted_cn"]),
                    "accepted_impaired":int(sums["accepted_impaired"]),
                    "rejected_target":int(sums["rejected_target"]),
                    "active_batches":int(sums["adapt_active_batches"]),
                    "blocked_single_class_batches":int(
                        sums["adapt_blocked_single_class_batches"]
                    ),
                },
            },
        )

        # ================================================================
        # CHECKPOINT SELECTION
        #
        # WARM y FULL se seleccionan INDEPENDIENTEMENTE.
        #
        # Ambos usan:
        #   source_validation / macro_f1
        #
        # Target NO interviene en checkpoint selection.
        # ================================================================
        
        current = float(
            epoch_metrics[
                "source_validation"
            ][
                "macro_f1"
            ]
        )
        
        
        # ================================================================
        # WARM
        # ================================================================

        if stage == "warm":
        
            if current > best_warm:
        
                best_warm = current
                best_warm_epoch = int(epoch)
        
                save_ckpt(
        
                    run_dir
                    / "checkpoint_best_warm_source_f1.pt",
        
                    model,
                    optimizer,
                    epoch,
                    best_warm,
                    history,
                    cfg,
                    disc,
                )
        
                print(
                    f"  -> NEW BEST WARM | "
                    f"epoch={epoch} | "
                    f"source-val macro-F1={best_warm:.4f}"
                )
        
        
        # ================================================================
        # FULL
        # ================================================================
        
        elif stage == "full":
        
            if current > best_full:
        
                best_full = current
                best_full_epoch = int(epoch)
        
                save_ckpt(
        
                    run_dir
                    / "checkpoint_best_full_source_f1.pt",
        
                    model,
                    optimizer,
                    epoch,
                    best_full,
                    history,
                    cfg,
                    disc,
                )
        
                print(
                    f"  -> NEW BEST FULL | "
                    f"epoch={epoch} | "
                    f"source-val macro-F1={best_full:.4f}"
                )
        

        # ================================================================
        # CHECKPOINT LAST
        #
        # El campo best_source_macro_f1 de checkpoint_last.pt no se utiliza
        # para seleccionar el resultado final. Solo lo conservamos por
        # compatibilidad con save_ckpt().
        # ================================================================
        
        best_for_last = (
            best_full
            if np.isfinite(best_full)
            else best_warm
        )
        
        save_ckpt(
            last,
            model,
            optimizer,
            epoch,
            best_for_last,
            history,
            cfg,
            disc,
        )

        pd.DataFrame(
            history
        ).to_csv(
            run_dir
            / "training_history.csv",
            index=False,
        )
        
        print(
            f"\n[{run_name}] {direction} fold={fold} epoch={epoch}"
        )
        print(
            " Source val F1:",
            round(epoch_metrics["source_validation"]["macro_f1"],4),
            "| BA:",
            round(epoch_metrics["source_validation"]["balanced_accuracy"],4),
            "| AUC:",
            round(epoch_metrics["source_validation"]["roc_auc"],4),
        )
        if method=="pada3dacb" and stage=="full":
            print(
                " PADA ramp:",round(pada_ramp,3),
                "| pseudo CN:",sums["accepted_cn"],
                "| pseudo Impaired:",sums["accepted_impaired"],
                "| blocked batches:",
                sums["adapt_blocked_single_class_batches"],
            )


    # ================================================================
    # EVALUACIÓN FINAL DE LOS DOS CHECKPOINTS
    # ================================================================
    
    warm_checkpoint = (
        run_dir
        / "checkpoint_best_warm_source_f1.pt"
    )
    
    full_checkpoint = (
        run_dir
        / "checkpoint_best_full_source_f1.pt"
    )
    
    
    if not warm_checkpoint.is_file():
    
        raise RuntimeError(
            "No existe "
            "checkpoint_best_warm_source_f1.pt"
        )
    
    
    if not full_checkpoint.is_file():
    
        raise RuntimeError(
            "No existe "
            "checkpoint_best_full_source_f1.pt. "
            "Los resultados oficiales requieren "
            "al menos una epoch FULL válida."
        )


    # ================================================================
    # 1. EVALUAR MEJOR WARM
    #
    # Se conserva como resultado diagnóstico.
    # ================================================================
    
    warm_p = load_ckpt(
        warm_checkpoint,
        model,
        device,
        discriminator=disc,
    )
    
    warm_metrics, warm_predictions = (
        evaluate_all_splits(
            model,
            loaders,
            device,
            roi_masks=masks,
            baseline=False,
        )
    )
    
    
    warm_payload = {
    
        "direction":
            direction,
    
        "method":
            method,

        "ablation":
            ablation,
    
        "fold":
            int(fold),
    
        "seed":
            int(seed),
    
        "phase":
            "warm",
    
        "best_epoch":
            int(
                warm_p["epoch"]
            ),
    
        "best_source_validation_macro_f1":
            float(
                warm_metrics[
                    "source_validation"
                ][
                    "macro_f1"
                ]
            ),
    
        "checkpoint":
            warm_checkpoint.name,
    
        "checkpoint_selection": {
    
            "phase":
                "warm",
    
            "split":
                "source_validation",
    
            "metric":
                "macro_f1",
    
            "target_labels_used_for_selection":
                False,
        },
    
        "metrics":
            warm_metrics,
    }
    
    
    (
        run_dir
        / "metrics_best_warm.json"
    ).write_text(
    
        json.dumps(
            warm_payload,
            indent=2,
        ),
    
        encoding="utf-8",
    )
    
    
    # ================================================================
    # GUARDAR PREDICCIONES WARM
    #
    # Se prefijan con warm_ para no confundirse con las oficiales.
    # ================================================================
    
    for (
        split_name,
        predictions,
    ) in warm_predictions.items():
    
        predictions.to_csv(
    
            run_dir
            / f"warm_{split_name}_predictions.csv",
    
            index=False,
        )
    
        prediction_errors = (
            predictions[
                predictions[
                    "is_error"
                ]
            ]
            .copy()
        )
    
        prediction_errors.to_csv(
    
            run_dir
            / f"warm_{split_name}_errors.csv",
    
            index=False,
        )


    # ================================================================
    # 2. EVALUAR MEJOR FULL
    #
    # ESTE SERÁ EL RESULTADO OFICIAL DEL EXPERIMENTO.
    # ================================================================
    
    full_p = load_ckpt(
        full_checkpoint,
        model,
        device,
        discriminator=disc,
    )
    
    full_metrics, full_predictions = (
        evaluate_all_splits(
            model,
            loaders,
            device,
            roi_masks=masks,
            baseline=False,
        )
    )
    
    
    full_payload = {
    
        "direction":
            direction,
    
        "method":
            method,
    
        "ablation":
            ablation,
    
        "fold":
            int(fold),
    
        "seed":
            int(seed),
    
        "phase":
            "full",
    
        "best_epoch":
            int(
                full_p["epoch"]
            ),
    
        "best_source_validation_macro_f1":
            float(
                full_metrics[
                    "source_validation"
                ][
                    "macro_f1"
                ]
            ),
    
        "checkpoint":
            full_checkpoint.name,
    
        "checkpoint_selection": {
    
            "phase":
                "full",
    
            "split":
                "source_validation",
    
            "metric":
                "macro_f1",
    
            "target_labels_used_for_selection":
                False,
        },
    
        "metrics":
            full_metrics,
    }
    
    
    # ================================================================
    # ARCHIVO EXPLÍCITO DEL MEJOR FULL
    # ================================================================

    (
        run_dir
        / "metrics_best_full.json"
    ).write_text(
    
        json.dumps(
            full_payload,
            indent=2,
        ),
    
        encoding="utf-8",
    )
    
    
    # ================================================================
    # metrics.json = RESULTADO OFICIAL
    #
    # MUY IMPORTANTE:
    # El archivo que leen las tablas/agregaciones seguirá llamándose
    # metrics.json, pero SIEMPRE corresponderá al mejor FULL.
    # ================================================================
    
    official_payload = {
    
        "direction":
            direction,
    
        "method":
            method,

        "ablation":
            ablation,
    
        "fold":
            int(fold),
    
        "seed":
            int(seed),
    
        # ------------------------------------------------------------
        # Información WARM
        # ------------------------------------------------------------
    
        "best_warm_epoch":
            int(
                warm_p["epoch"]
            ),
    
        "best_warm_source_validation_macro_f1":
            float(
                warm_metrics[
                    "source_validation"
                ][
                    "macro_f1"
                ]
            ),
    
        "warm_checkpoint":
            warm_checkpoint.name,


        # ------------------------------------------------------------
        # Información FULL
        # ------------------------------------------------------------
    
        "best_full_epoch":
            int(
                full_p["epoch"]
            ),
    
        "best_full_source_validation_macro_f1":
            float(
                full_metrics[
                    "source_validation"
                ][
                    "macro_f1"
                ]
            ),
    
        "full_checkpoint":
            full_checkpoint.name,
    
    
        # ------------------------------------------------------------
        # Compatibilidad
        #
        # best_epoch SIEMPRE significa best FULL.
        # ------------------------------------------------------------
    
        "best_epoch":
            int(
                full_p["epoch"]
            ),

        "official_phase":
            "full",
    
        "official_checkpoint":
            full_checkpoint.name,
    
        "checkpoint_selection": {
    
            "phase":
                "full",
    
            "split":
                "source_validation",
    
            "metric":
                "macro_f1",
    
            "target_labels_used_for_selection":
                False,
        },
    
    
        # ------------------------------------------------------------
        # IMPORTANTE:
        #
        # Estas son EXCLUSIVAMENTE las métricas FULL.
        # ------------------------------------------------------------
    
        "metrics":
            full_metrics,
    }
    
    
    (
        run_dir
        / "metrics.json"
    ).write_text(
    
        json.dumps(
            official_payload,
            indent=2,
        ),
    
        encoding="utf-8",
    )
    
    
    # ================================================================
    # PREDICCIONES OFICIALES = FULL
    #
    # Los archivos sin prefijo siempre corresponden a FULL.
    # ================================================================
    
    for (
        split_name,
        predictions,
    ) in full_predictions.items():
    
        predictions.to_csv(
    
            run_dir
            / f"{split_name}_predictions.csv",
    
            index=False,
        )
    
        prediction_errors = (
            predictions[
                predictions[
                    "is_error"
                ]
            ]
            .copy()
        )
    
        prediction_errors.to_csv(
    
            run_dir
            / f"{split_name}_errors.csv",
    
            index=False,
        )


    # ================================================================
    # THRESHOLD CALIBRADO SOLO EN SOURCE VALIDATION (DIAGNÓSTICO)
    #
    # No reemplaza metrics.json ni interviene en checkpoint selection.
    # ================================================================

    threshold_info=find_source_validation_threshold(
        full_predictions["source_validation"]
    )
    source_calibrated_threshold=float(threshold_info["threshold"])

    calibrated_metrics={}
    for split_name,predictions in full_predictions.items():
        calibrated_metrics[split_name]=metrics_from_prediction_table(
            predictions,
            source_calibrated_threshold,
        )

        calibrated_predictions=predictions.copy()
        calibrated_predictions["source_calibrated_threshold"]=(
            source_calibrated_threshold
        )
        calibrated_predictions["y_pred_source_calibrated"]=(
            calibrated_predictions["prob_impaired"]
            .ge(source_calibrated_threshold)
            .astype(int)
        )
        calibrated_predictions["predicted_label_source_calibrated"]=(
            calibrated_predictions["y_pred_source_calibrated"]
            .map({0:"CN",1:"Impaired"})
        )
        calibrated_predictions.to_csv(
            run_dir/f"{split_name}_predictions_source_calibrated.csv",
            index=False,
        )

    calibration_payload={
        "selection_split":"source_validation",
        "selection_objective":"balanced_accuracy_then_macro_f1",
        "target_labels_used_for_threshold_selection":False,
        "threshold":source_calibrated_threshold,
        "source_validation_selection_metrics":threshold_info,
        "metrics":calibrated_metrics,
    }
    (run_dir/"metrics_source_calibrated_threshold.json").write_text(
        json.dumps(calibration_payload,indent=2),
        encoding="utf-8",
    )

    print(
        " Source-calibrated threshold:",
        round(source_calibrated_threshold,4),
        "| source-val BA:",
        round(calibrated_metrics["source_validation"]["balanced_accuracy"],4),
        "| target-val BA [diagnostic]:",
        round(calibrated_metrics["target_validation"]["balanced_accuracy"],4),
    )


    # ================================================================
    # COMPARACIÓN DIRECTA WARM vs FULL
    # ================================================================
    
    phase_comparison = {
    
        "warm": {
    
            "epoch":
                int(
                    warm_p["epoch"]
                ),
    
            "source_validation_macro_f1":
                float(
                    warm_metrics[
                        "source_validation"
                    ][
                        "macro_f1"
                    ]
                ),
    
            "target_validation_macro_f1":
                float(
                    warm_metrics[
                        "target_validation"
                    ][
                        "macro_f1"
                    ]
                ),
    
            "target_validation_accuracy":
                float(
                    warm_metrics[
                        "target_validation"
                    ][
                        "accuracy"
                    ]
                ),
        },
    
    
        "full": {
    
            "epoch":
                int(
                    full_p["epoch"]
                ),
    
            "source_validation_macro_f1":
                float(
                    full_metrics[
                        "source_validation"
                    ][
                        "macro_f1"
                    ]
                ),
    
            "target_validation_macro_f1":
                float(
                    full_metrics[
                        "target_validation"
                    ][
                        "macro_f1"
                    ]
                ),
    
            "target_validation_accuracy":
                float(
                    full_metrics[
                        "target_validation"
                    ][
                        "accuracy"
                    ]
                ),
        },
    }
    
    
    (
        run_dir
        / "warm_vs_full.json"
    ).write_text(
    
        json.dumps(
            phase_comparison,
            indent=2,
        ),
    
        encoding="utf-8",
    )
    
    
    # ================================================================
    # COMPLETED.JSON
    #
    # "metrics" = FULL para que results_df y las agregaciones utilicen
    # exclusivamente la fase FULL.
    # ================================================================
    
    completed = {
    
        "status":
            "COMPLETED",
    
        "direction":
            direction,
    
        "method":
            method,
    
        "ablation":
            ablation,

        "run_name":
            run_name,
    
        "fold":
            int(fold),
    
        "seed":
            int(seed),
    
    
        # WARM
        "best_warm_epoch":
            int(
                warm_p["epoch"]
            ),
    
        "best_warm_source_validation_macro_f1":
            float(
                warm_metrics[
                    "source_validation"
                ][
                    "macro_f1"
                ]
            ),
    
        "warm_checkpoint":
            warm_checkpoint.name,
    
    
        # FULL
        "best_full_epoch":
            int(
                full_p["epoch"]
            ),
    
        "best_full_source_validation_macro_f1":
            float(
                full_metrics[
                    "source_validation"
                ][
                    "macro_f1"
                ]
            ),
    
        "full_checkpoint":
            full_checkpoint.name,
    
    
        # Compatibilidad:
        # best_epoch = mejor FULL
        "best_epoch":
            int(
                full_p["epoch"]
            ),
    
        "official_phase":
            "full",
    
        "official_checkpoint":
            full_checkpoint.name,
    
    
        # MUY IMPORTANTE:
        # results_df recibirá las métricas FULL.
        "source_calibrated_threshold":
            float(source_calibrated_threshold),

        "metrics":
            full_metrics,
    
        "warm_metrics":
            warm_metrics,
    
        "run_dir":
            str(run_dir),
    
        "reused":
            False,
    }
    
    
    marker.write_text(
    
        json.dumps(
            completed,
            indent=2,
        ),
    
        encoding="utf-8",
    )
    del model,core,loaders,optimizer
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()
    return completed

## 9. Entrenador AAGN / Faster-SNN

In [ ]:
def run_baseline(direction,fold,seed,name):
    seed_all(seed); device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    run_dir=RUNS_ROOT/direction/name/f"seed_{seed}"/f"fold_{fold}"; run_dir.mkdir(parents=True,exist_ok=True)
    marker=run_dir/"COMPLETED.json"
    if marker.is_file() and not EFFECTIVE_TRAINING["overwrite"]:
        x=json.loads(marker.read_text()); x["reused"]=True; return x

    frames=load_fold_frames(direction,fold); loaders=build_loaders(frames,seed,pada=False)
    model=build_binary_baseline(name).to(device); cfg=EFFECTIVE_BASELINE_TRAINING
    opt=torch.optim.AdamW(model.parameters(),lr=cfg["learning_rate"],weight_decay=cfg["weight_decay"])
    ce=nn.CrossEntropyLoss(); amp=bool(cfg["mixed_precision"] and device.type=="cuda")
    scaler=torch.amp.GradScaler("cuda",enabled=amp); history=[]; best=-np.inf
    rcfg={"direction":direction,"fold":fold,"seed":seed,"baseline":name,"params":BASELINES[name],"training":cfg}
    (run_dir/"resolved_config.json").write_text(json.dumps(rcfg,indent=2),encoding="utf-8")

    for epoch in range(1,int(cfg["epochs"])+1):
        model.train(); total=0.; n=0
        for b in tqdm(loaders["source_train"],desc=f"{name} {direction} f{fold} e{epoch}/{cfg['epochs']}",leave=False):
            x=b["x"].to(device,non_blocking=True); y=b["y"].to(device,non_blocking=True); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type,enabled=amp):
                o=model(x); logits=o["logits"] if isinstance(o,dict) else o; loss=ce(logits,y)
            if scaler.is_enabled():
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(),cfg["gradient_clip_norm"]); scaler.step(opt); scaler.update()
            else:
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),cfg["gradient_clip_norm"]); opt.step()
            total+=float(loss.detach().cpu()); n+=1
        epoch_metrics, _ = (
            evaluate_all_splits(
                model,
                loaders,
                device,
                baseline=True,
            )
        )
        
        row = {
        
            "epoch":
                int(epoch),
        
            "train_loss":
                total
                / max(n, 1),
        
            **flatten_split_metrics(
                epoch_metrics
            ),
        }
        
        history.append(
            row
        )
        
        append_jsonl(
            run_dir
            / "epoch_metrics.jsonl",
        
            {
                "epoch":
                    int(epoch),
        
                "metrics":
                    epoch_metrics,
            },
        )
        
        current = float(
            epoch_metrics[
                "source_validation"
            ][
                "macro_f1"
            ]
        )
        if current>best:
            best=current; save_ckpt(run_dir/"checkpoint_best_source_f1.pt",model,opt,epoch,best,history,rcfg)
        save_ckpt(run_dir/"checkpoint_last.pt",model,opt,epoch,best,history,rcfg)
        pd.DataFrame(history).to_csv(run_dir/"training_history.csv",index=False)

    bp=load_ckpt(run_dir/"checkpoint_best_source_f1.pt",model,device)
    final_metrics, final_predictions = (
        evaluate_all_splits(
            model,
            loaders,
            device,
            roi_masks=masks,
            baseline=True,
        )
    )
    for split_name, predictions in (
        final_predictions.items()
    ):
    
        # ============================================================
        # TODAS LAS PREDICCIONES
        # ============================================================
    
        predictions.to_csv(
    
            run_dir
            / f"{split_name}_predictions.csv",
    
            index=False,
        )
    
        # ============================================================
        # SOLO ERRORES
        # ============================================================
    
        prediction_errors = (
            predictions[
                predictions[
                    "is_error"
                ]
            ]
            .copy()
        )

        prediction_errors.to_csv(
    
            run_dir
            / f"{split_name}_errors.csv",
    
            index=False,
        )
    metrics_payload = {

        "direction":
            direction,
    
        "method":
            method,
    
        "ablation":
            ablation,
    
        "fold":
            int(fold),
    
        "seed":
            int(seed),
    
        "best_epoch":
            int(
                bp["epoch"]
            ),
    
        "checkpoint_selection": {
            "split":
                "source_validation",
    
            "metric":
                "macro_f1",
    
            "target_labels_used_for_selection":
                False,
        },
    
        "metrics":
            final_metrics,
    }
    (
        run_dir
        / "metrics.json"
    ).write_text(
    
        json.dumps(
            metrics_payload,
            indent=2,
        ),
    
        encoding="utf-8",
    )
    completed={"status":"COMPLETED","direction":direction,"method":name,"ablation":None,"run_name":name,
               "fold":fold,"seed":seed,"best_epoch":int(bp["epoch"]),
               "metrics": final_metrics,"run_dir":str(run_dir),"reused":False}
    marker.write_text(json.dumps(completed,indent=2),encoding="utf-8")
    del model,opt,loaders
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()
    return completed

### Diagnóstico esperado

Antes de correr la matriz completa, revisar:

- `target_pseudo_accepted_cn`
- `target_pseudo_accepted_impaired`
- `adapt_blocked_single_class_batches`
- `pada_adaptation_ramp`
- `source_validation/balanced_accuracy`
- `source_validation/roc_auc`

Si PADA empieza a aceptar pseudo-labels de ambas clases y `specificity`/`recall_cn`
dejan de ser cero en la evaluación final, ya tiene sentido escalar a los cinco folds.


## 10. Previsualizar la matriz de experimentos

In [ ]:
runs=[]
for d in DIRECTIONS:
    for s in EFFECTIVE_SEEDS:
        for f in EFFECTIVE_FOLDS:
            for m,on in METHODS_TO_RUN.items():
                if on:
                    runs.append({"direction":d,"seed":s,"fold":f,"kind":"baseline" if m in {"aagn","faster_snn"} else "pada",
                                 "method":m,"ablation":None,"run_name":m})
            for a,on in ABLATIONS_TO_RUN.items():
                if on:
                    runs.append({"direction":d,"seed":s,"fold":f,"kind":"pada","method":"pada3dacb",
                                 "ablation":a,"run_name":f"ablation_{a}"})
run_matrix=pd.DataFrame(runs)
display(run_matrix)
print("Total runs:",len(run_matrix))

## 11. Smoke test de contratos

In [ ]:
if len(run_matrix):
    r=run_matrix.iloc[0]; frames=load_fold_frames(r.direction,int(r.fold))
    needs_pada=bool((run_matrix.kind=="pada").any())
    loaders=build_loaders(frames,int(r.seed),pada=needs_pada)
    tb=next(iter(loaders["target_adaptation"]))
    if set(tb)!={"x","subject_id","subject_hash","cohort"}:
        raise RuntimeError(f"Target adaptation firewall inválido: {sorted(tb)}")
    print("✓ target adaptation:",sorted(tb))
    if needs_pada:
        sb=next(iter(loaders["source_train"]))
        print("x:",tuple(sb["x"].shape),"c_target:",tuple(sb["c_target"].shape),"g_bar:",tuple(sb["g_bar"].shape))
        device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
        m=build_pada().to(device); masks=FEATURE_ROI_MASKS.to(device)
        with torch.no_grad(): out=m(sb["x"].to(device),masks)
        print("concept logits:",tuple(out.concept_logits.shape),"concepts:",tuple(out.concepts.shape))
        if out.concept_logits.shape[-1]!=2: raise RuntimeError("Head no binario")
        del m,masks,out
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    del loaders
    gc.collect()
    print("✓ contratos OK")

## 12. Ejecutar la matriz

In [ ]:
results=[]; errors=[]
if EXECUTE_TRAINING:
    RUNS_ROOT.mkdir(parents=True,exist_ok=True)
    for r in run_matrix.itertuples(index=False):
        print("\n"+"="*90)
        print(r.direction,r.run_name,"seed",r.seed,"fold",r.fold)
        print("="*90)
        try:
            row=readiness_df[readiness_df.direction==r.direction].iloc[0]
            if r.kind=="pada" and not bool(row.pada_ready):
                raise RuntimeError("Faltan conceptos/Jacobians del source para PADA.")
            if r.kind=="baseline" and not bool(row.baseline_ready):
                raise RuntimeError("Faltan MRI para baseline.")
            if r.kind=="baseline":
                res=run_baseline(r.direction,int(r.fold),int(r.seed),r.method)
            else:
                res=run_pada(r.direction,int(r.fold),int(r.seed),r.method,r.ablation)
            results.append(res)
        except Exception as exc:

            full_traceback = (
                traceback.format_exc()
            )
        
            error_payload = {
        
                "timestamp":
                    pd.Timestamp.utcnow().isoformat(),
        
                "direction":
                    r.direction,
        
                "run_name":
                    r.run_name,
        
                "method":
                    r.method,
        
                "ablation":
                    r.ablation,
        
                "fold":
                    int(r.fold),
        
                "seed":
                    int(r.seed),
        
                "error_type":
                    type(exc).__name__,

                "error":
                    str(exc),
        
                "traceback":
                    full_traceback,
            }
        
            errors.append(
                error_payload
            )
        
            # ------------------------------------------------------------
            # JSONL inmediato.
            #
            # Incluso si Kaggle se detiene posteriormente,
            # los errores anteriores ya quedaron registrados.
            # ------------------------------------------------------------
        
            append_jsonl(
        
                RUNS_ROOT
                / "experiment_errors_full.jsonl",
        
                error_payload,
            )
        
            print()
            print("=" * 80)
            print("ERROR DE ENTRENAMIENTO")
            print("=" * 80)

            print(
                full_traceback
            )
        
            if not EFFECTIVE_TRAINING[
                "continue_on_error"
            ]:
        
                raise
        finally:
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            gc.collect()
else:
    print("EXECUTE_TRAINING=False")

results_df=pd.DataFrame(results); errors_df=pd.DataFrame(errors)
if len(results_df):
    results_df.to_csv(RUNS_ROOT/"experiment_summary.csv",index=False); display(results_df)
if len(errors_df):
    errors_df.to_csv(RUNS_ROOT/"experiment_errors.csv",index=False); display(errors_df)

## 13. Agregación de folds/seeds y tabla de ablaciones

In [ ]:
# ================================================================
# AGREGACIÓN ROBUSTA DE RESULTADOS
#
# Compatible con:
# - esquema antiguo:
#       source_validation_macro_f1
#       target_evaluation_macro_f1
#       target_evaluation_accuracy
#
# - esquema nuevo:
#       metrics.json
#           source_train
#           source_validation
#           target_train_diagnostic
#           target_validation
#
# También acepta target_evaluation por compatibilidad.
# ================================================================

if len(results_df):

    results_df = results_df.copy()

    print("=" * 80)
    print("COLUMNAS DISPONIBLES EN results_df")
    print("=" * 80)

    print(
        results_df.columns.tolist()
    )


    # ============================================================
    # RESOLVER DIRECTORIO DE CADA RUN
    # ============================================================

    def resolve_run_dir(row):

        # Si results_df ya contiene run_dir, utilizarlo.
        if (
            "run_dir" in row.index
            and
            pd.notna(row["run_dir"])
            and
            str(row["run_dir"]).strip()
        ):

            return Path(
                str(row["run_dir"])
            )


        # Reconstrucción determinista.
        return (
            RUNS_ROOT
            / str(row["direction"])
            / str(row["run_name"])
            / f"seed_{int(row['seed'])}"
            / f"fold_{int(row['fold'])}"
        )


    # ============================================================
    # CARGAR metrics.json
    # ============================================================

    def load_run_metrics(row):

        run_dir = resolve_run_dir(
            row
        )

        metrics_path = (
            run_dir
            / "metrics.json"
        )

        if not metrics_path.is_file():

            return {}


        try:

            with open(
                metrics_path,
                "r",
                encoding="utf-8",
            ) as f:

                payload = json.load(
                    f
                )


        except Exception as exc:

            print(
                "WARNING: no se pudo leer",
                metrics_path,
                "->",
                type(exc).__name__,
                str(exc),
            )

            return {}


        # --------------------------------------------------------
        # Esquema nuevo:
        #
        # {
        #     "metrics": {
        #         "source_train": {...},
        #         ...
        #     }
        # }
        #
        # Esquema anterior:
        #
        # {
        #     "source_validation": {...},
        #     "target_evaluation": {...}
        # }
        # --------------------------------------------------------

        if (
            isinstance(
                payload.get("metrics"),
                dict,
            )
        ):

            return payload["metrics"]


        return payload


    # ============================================================
    # OBTENER UNA MÉTRICA DE FORMA ROBUSTA
    # ============================================================

    def extract_metric(
        row,
        *,
        old_columns=(),
        split_names=(),
        metric_name,
    ):

        # --------------------------------------------------------
        # 1. Intentar columnas ya presentes en results_df
        # --------------------------------------------------------

        for column in old_columns:

            if (
                column in row.index
                and
                pd.notna(row[column])
            ):

                try:

                    return float(
                        row[column]
                    )

                except (
                    TypeError,
                    ValueError,
                ):

                    pass


        # --------------------------------------------------------
        # 2. Intentar nombres estilo:
        #
        # source_validation/macro_f1
        # --------------------------------------------------------

        for split_name in split_names:

            flat_name = (
                f"{split_name}/{metric_name}"
            )

            if (
                flat_name in row.index
                and
                pd.notna(
                    row[flat_name]
                )
            ):

                try:

                    return float(
                        row[flat_name]
                    )

                except (
                    TypeError,
                    ValueError,
                ):

                    pass


        # --------------------------------------------------------
        # 3. Leer metrics.json
        # --------------------------------------------------------

        metrics = load_run_metrics(
            row
        )


        for split_name in split_names:

            split = metrics.get(
                split_name
            )

            if not isinstance(
                split,
                dict,
            ):

                continue


            value = split.get(
                metric_name
            )


            if value is None:

                continue


            try:

                return float(
                    value
                )

            except (
                TypeError,
                ValueError,
            ):

                continue


        return np.nan


    # ============================================================
    # NORMALIZAR COLUMNAS PRINCIPALES
    # ================================================================

    results_df[
        "source_validation_macro_f1"
    ] = results_df.apply(

        lambda row: extract_metric(

            row,

            old_columns=(
                "source_validation_macro_f1",
            ),

            split_names=(
                "source_validation",
            ),

            metric_name="macro_f1",
        ),

        axis=1,
    )


    results_df[
        "target_validation_macro_f1"
    ] = results_df.apply(

        lambda row: extract_metric(

            row,

            old_columns=(
                "target_evaluation_macro_f1",
                "target_validation_macro_f1",
            ),

            split_names=(
                "target_validation",
                "target_evaluation",
            ),

            metric_name="macro_f1",
        ),

        axis=1,
    )


    results_df[
        "target_validation_accuracy"
    ] = results_df.apply(

        lambda row: extract_metric(

            row,

            old_columns=(
                "target_evaluation_accuracy",
                "target_validation_accuracy",
            ),

            split_names=(
                "target_validation",
                "target_evaluation",
            ),

            metric_name="accuracy",
        ),

        axis=1,
    )


    # ============================================================
    # TAMBIÉN RECUPERAR TRAIN
    # ================================================================

    results_df[
        "source_train_macro_f1"
    ] = results_df.apply(

        lambda row: extract_metric(

            row,

            old_columns=(
                "source_train_macro_f1",
            ),

            split_names=(
                "source_train",
            ),

            metric_name="macro_f1",
        ),

        axis=1,
    )


    results_df[
        "source_train_accuracy"
    ] = results_df.apply(

        lambda row: extract_metric(

            row,

            old_columns=(
                "source_train_accuracy",
            ),

            split_names=(
                "source_train",
            ),

            metric_name="accuracy",
        ),

        axis=1,
    )


    results_df[
        "target_train_macro_f1"
    ] = results_df.apply(

        lambda row: extract_metric(

            row,

            old_columns=(
                "target_train_macro_f1",
            ),

            split_names=(
                "target_train_diagnostic",
                "target_train",
            ),

            metric_name="macro_f1",
        ),

        axis=1,
    )


    results_df[
        "target_train_accuracy"
    ] = results_df.apply(

        lambda row: extract_metric(

            row,

            old_columns=(
                "target_train_accuracy",
            ),

            split_names=(
                "target_train_diagnostic",
                "target_train",
            ),

            metric_name="accuracy",
        ),

        axis=1,
    )


    # ============================================================
    # QC
    # ================================================================

    metric_columns = [

        "source_train_macro_f1",
        "source_train_accuracy",

        "source_validation_macro_f1",

        "target_train_macro_f1",
        "target_train_accuracy",

        "target_validation_macro_f1",
        "target_validation_accuracy",
    ]


    print()
    print("=" * 80)
    print("MÉTRICAS RECUPERADAS")
    print("=" * 80)


    display(

        results_df[
            [
                "direction",
                "run_name",
                "fold",
                "seed",
                *metric_columns,
            ]
        ]

    )


    # ============================================================
    # REPORTAR MÉTRICAS NO RECUPERADAS
    # ================================================================

    missing_summary = (

        results_df[
            metric_columns
        ]
        .isna()
        .sum()
        .rename(
            "missing"
        )
        .reset_index()
        .rename(
            columns={
                "index":
                    "metric"
            }
        )

    )


    print()
    print(
        "Valores faltantes por métrica:"
    )

    display(
        missing_summary
    )


    # ============================================================
    # AGREGACIÓN GENERAL
    # ================================================================

    agg = (

        results_df

        .groupby(
            [
                "direction",
                "run_name",
            ],
            as_index=False,
        )

        .agg(

            n_runs=(
                "fold",
                "size",
            ),


            # ====================================================
            # SOURCE TRAIN
            # ====================================================

            source_train_macro_f1_mean=(
                "source_train_macro_f1",
                "mean",
            ),

            source_train_macro_f1_std=(
                "source_train_macro_f1",
                "std",
            ),

            source_train_accuracy_mean=(
                "source_train_accuracy",
                "mean",
            ),

            source_train_accuracy_std=(
                "source_train_accuracy",
                "std",
            ),


            # ====================================================
            # SOURCE VALIDATION
            # ====================================================

            source_val_macro_f1_mean=(
                "source_validation_macro_f1",
                "mean",
            ),

            source_val_macro_f1_std=(
                "source_validation_macro_f1",
                "std",
            ),


            # ====================================================
            # TARGET TRAIN DIAGNOSTIC
            # ====================================================

            target_train_macro_f1_mean=(
                "target_train_macro_f1",
                "mean",
            ),

            target_train_macro_f1_std=(
                "target_train_macro_f1",
                "std",
            ),

            target_train_accuracy_mean=(
                "target_train_accuracy",
                "mean",
            ),

            target_train_accuracy_std=(
                "target_train_accuracy",
                "std",
            ),


            # ====================================================
            # TARGET VALIDATION
            # ====================================================

            target_val_macro_f1_mean=(
                "target_validation_macro_f1",
                "mean",
            ),

            target_val_macro_f1_std=(
                "target_validation_macro_f1",
                "std",
            ),

            target_val_accuracy_mean=(
                "target_validation_accuracy",
                "mean",
            ),

            target_val_accuracy_std=(
                "target_validation_accuracy",
                "std",
            ),
        )

    )


    agg.to_csv(

        RUNS_ROOT
        / "aggregated_results.csv",

        index=False,
    )


    print()
    print("=" * 80)
    print("RESULTADOS AGREGADOS")
    print("=" * 80)


    display(

        agg.sort_values(

            [
                "direction",
                "target_val_macro_f1_mean",
            ],

            ascending=[
                True,
                False,
            ],

            na_position="last",
        )

    )


    # ============================================================
    # RESUMEN DE ABLACIONES
    # ================================================================

    abl = results_df[

        (
            results_df[
                "run_name"
            ]
            == "pada3dacb"
        )

        |

        (
            results_df[
                "run_name"
            ]
            .astype(str)
            .str.startswith(
                "ablation_"
            )
        )

    ].copy()


    if len(abl):

        abl_sum = (

            abl

            .groupby(
                [
                    "direction",
                    "run_name",
                ],
                as_index=False,
            )

            .agg(

                n_runs=(
                    "fold",
                    "size",
                ),

                source_train_macro_f1=(
                    "source_train_macro_f1",
                    "mean",
                ),

                source_val_macro_f1=(
                    "source_validation_macro_f1",
                    "mean",
                ),

                target_train_macro_f1=(
                    "target_train_macro_f1",
                    "mean",
                ),

                target_val_macro_f1=(
                    "target_validation_macro_f1",
                    "mean",
                ),

                target_val_accuracy=(
                    "target_validation_accuracy",
                    "mean",
                ),
            )

        )


        abl_sum.to_csv(

            RUNS_ROOT
            / "ablation_summary.csv",

            index=False,
        )


        print()
        print("=" * 80)
        print("RESUMEN DE ABLACIONES")
        print("=" * 80)


        display(

            abl_sum.sort_values(

                [
                    "direction",
                    "target_val_macro_f1",
                ],

                ascending=[
                    True,
                    False,
                ],

                na_position="last",
            )

        )


    # ============================================================
    # GUARDAR RESULTS_DF NORMALIZADO
    # ================================================================

    results_df.to_csv(

        RUNS_ROOT
        / "experiment_summary_normalized.csv",

        index=False,
    )


else:

    print(
        "No hay resultados todavía."
    )

In [ ]:
prediction_error_files = list(
    RUNS_ROOT.rglob(
        "*_errors.csv"
    )
)

all_prediction_errors = []

for path in (
    prediction_error_files
):

    try:

        current = pd.read_csv(
            path
        )

        if len(current) == 0:
            continue

        current[
            "source_file"
        ] = str(
            path.relative_to(
                RUNS_ROOT
            )
        )

        all_prediction_errors.append(
            current
        )

    except Exception as exc:

        print(
            "No se pudo leer:",
            path,
            exc,
        )


if all_prediction_errors:

    all_prediction_errors_df = (
        pd.concat(
            all_prediction_errors,
            ignore_index=True,
        )
    )

    all_prediction_errors_df.to_csv(

        RUNS_ROOT
        / "all_prediction_errors.csv",

        index=False,
    )

    display(
        all_prediction_errors_df
    )

else:

    all_prediction_errors_df = (
        pd.DataFrame()
    )

    print(
        "No se encontraron errores predictivos."
    )

metric_files = list(
    RUNS_ROOT.rglob(
        "metrics.json"
    )
)

all_metric_rows = []


for path in metric_files:

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        payload = json.load(
            f
        )

    metrics = payload.get(
        "metrics",
        payload,
    )

    for split_name in [
        "source_train",
        "source_validation",
        "target_train_diagnostic",
        "target_validation",
    ]:

        if split_name not in metrics:
            continue

        current = metrics[
            split_name
        ]

        row = {

            "run_path":
                str(
                    path.parent.relative_to(
                        RUNS_ROOT
                    )
                ),

            "direction":
                payload.get(
                    "direction"
                ),

            "method":
                payload.get(
                    "method",
                    payload.get(
                        "baseline"
                    ),
                ),

            "ablation":
                payload.get(
                    "ablation"
                ),

            "fold":
                payload.get(
                    "fold"
                ),

            "seed":
                payload.get(
                    "seed"
                ),

            "best_epoch":
                payload.get(
                    "best_epoch"
                ),

            "split":
                split_name,
        }

        for metric, value in (
            current.items()
        ):

            if metric == (
                "confusion_matrix"
            ):

                row[
                    "confusion_matrix"
                ] = json.dumps(
                    value
                )

            else:

                row[
                    metric
                ] = value

        all_metric_rows.append(
            row
        )


all_metrics_df = pd.DataFrame(
    all_metric_rows
)

all_metrics_df.to_csv(

    RUNS_ROOT
    / "all_final_metrics.csv",

    index=False,
)

display(
    all_metrics_df
)



## Salidas

Además de los archivos originales, cada run diagnóstico guarda:

```text
training_history.csv
epoch_metrics.jsonl
metrics.json
metrics_source_calibrated_threshold.json
<split>_predictions.csv
<split>_predictions_source_calibrated.csv
```

`metrics.json` conserva la evaluación oficial del mejor checkpoint FULL,
seleccionado por `source_validation/macro_f1`.

`metrics_source_calibrated_threshold.json` es solo un diagnóstico adicional:
el threshold se ajusta con `source_validation` y nunca con etiquetas target.
